# MAESTRO - Fine-tuning sur TreeSatAI (8 classes)

Entrainement de la tete de classification MAESTRO sur le jeu de donnees TreeSatAI
avec 8 classes d'essences regroupees.

**Pre-requis** : Activer le GPU dans Runtime > Change runtime type > T4 GPU

In [ ]:
# Verifier le GPU
import torch
print(f'PyTorch {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} Go')

In [ ]:
# Installer les dependances
!pip install -q rasterio h5py huggingface_hub safetensors

In [ ]:
# Cloner le depot
!git clone -b cleanup/code-review https://github.com/pobsteta/maestro_nemeton.git
%cd maestro_nemeton

In [ ]:
# Telecharger le checkpoint MAESTRO pre-entraine
from huggingface_hub import hf_hub_download

checkpoint_path = hf_hub_download(
    'IGNF/MAESTRO_FLAIR-HUB_base',
    'MAESTRO_FLAIR-HUB_base/checkpoints/pretrain-epoch=99.ckpt'
)
print(f'Checkpoint: {checkpoint_path}')

In [ ]:
# Telecharger et preparer les donnees TreeSatAI
# (~16 Go aerial, peut prendre 10-15 min)
import sys
sys.path.insert(0, 'inst/python')
from train_treesatai import preparer_donnees

labels, train_files, val_files, test_files = preparer_donnees('data/treesatai')
print(f'Train: {len(train_files)}, Val: {len(val_files)}, Test: {len(test_files)}')

In [ ]:
# Lancer l'entrainement (backbone gele, ~15 min sur T4)
!python inst/python/train_treesatai.py \
    --checkpoint {checkpoint_path} \
    --data-dir data/treesatai \
    --output-dir outputs/training \
    --modalites aerial \
    --epochs 30 \
    --batch-size 64 \
    --lr 1e-3 \
    --gpu \
    --workers 2

In [ ]:
# (Optionnel) Fine-tuning complet - decommenter pour lancer
# Plus long (~45 min) mais potentiellement meilleur

# !python inst/python/train_treesatai.py \
#     --checkpoint {checkpoint_path} \
#     --data-dir data/treesatai \
#     --output-dir outputs/training_unfreeze \
#     --modalites aerial \
#     --epochs 30 \
#     --batch-size 32 \
#     --lr 5e-4 \
#     --gpu \
#     --unfreeze \
#     --workers 2

In [ ]:
# Voir les resultats
import os
for f in os.listdir('outputs/training'):
    size = os.path.getsize(f'outputs/training/{f}') / 1e6
    print(f'{f}: {size:.1f} Mo')

In [ ]:
# Charger et inspecter le meilleur modele
chk = torch.load('outputs/training/maestro_treesatai_best.pt', map_location='cpu')
print(f"Epoch: {chk['epoch']}")
print(f"Val accuracy: {chk['val_acc']:.3f}")
print(f"Classes: {chk['classes']}")

In [ ]:
# Telecharger le modele entraine sur ton PC
from google.colab import files
files.download('outputs/training/maestro_treesatai_best.pt')